In [2]:
import pandas as pd 
import os 
from datetime import datetime 

In [3]:
df_fr_gas = pd.read_csv("gas_fr_plants.csv")
df_gas = pd.read_csv("gas_plants.csv")
df_wind = pd.read_csv("wind_plants.csv")

In [4]:
datasets = {
    "gas_fr_plants.csv": df_fr_gas,
    "gas_plants.csv": df_gas,
    "wind_plants.csv": df_wind
}

# Structure check
I checked the shape, first and last rows, column names, and data types of each dataset to understand the overall structure before applying detailed validation checks.

This check showed that `gas_fr_plants.csv` contains multiple fully empty rows at the end of the file. It also showed that the `Volume` column in `gas_plants.csv` is stored as `object` rather than a numeric type, which suggests that the column may contain mixed or invalid values and requires further investigation also all three datasets have trailing whitespace in the `Country` column name.

These issues matter because column whitespace can break column references in the pipeline, fully empty rows should not be stored in `database.csv` because they contain no useful information and `Volume` must be numeric because it is used for the final aggregations and calculations.In order to solve these issues pipeline will strip whitespace from column names, remove fully empty rows, and safely convert `Volume` to numeric after investigating the invalid or mixed values.


In [6]:
def basic_structure_check(df):
    
    print("BASIC STRUCTURE CHECK")
    print("=" * 50)
    
    print("\nShape:")
    print(df.shape)
    
    print("\nFirst 5 rows:")
    display(df.head())
    
    print("\nLast 5 rows:")
    display(df.tail())
    
    print("\nColumn names:")
    print(df.columns.tolist())
    
    print("\nData types:")
    print(df.dtypes)

In [7]:
for name, df in datasets.items():
    print("\n")
    print("#" * 80)
    print(f"BASIC STRUCTURE CHECK: {name}")
    print("#" * 80)
    
    basic_structure_check(df)



################################################################################
BASIC STRUCTURE CHECK: gas_fr_plants.csv
################################################################################
BASIC STRUCTURE CHECK

Shape:
(962, 5)

First 5 rows:


,Date,Country,Technology,SiteName,Volume
0,01/01/2024,FR,Gas,Blenod-5,6753.0
1,02/01/2024,FR,Gas,Blenod-5,3896.0
2,03/01/2024,FR,Gas,Blenod-5,3636.0
3,04/01/2024,FR,Gas,Blenod-5,5138.0
4,05/01/2024,FR,Gas,Blenod-5,5265.0



Last 5 rows:


,Date,Country,Technology,SiteName,Volume
957,NaN,NaN,NaN,NaN,NaN
958,NaN,NaN,NaN,NaN,NaN
959,NaN,NaN,NaN,NaN,NaN
960,NaN,NaN,NaN,NaN,NaN
961,NaN,NaN,NaN,NaN,NaN



Column names:
['Date', 'Country ', 'Technology', 'SiteName', 'Volume']

Data types:
Date           object
Country        object
Technology     object
SiteName       object
Volume        float64
dtype: object


################################################################################
BASIC STRUCTURE CHECK: gas_plants.csv
################################################################################
BASIC STRUCTURE CHECK

Shape:
(962, 5)

First 5 rows:


...


Column names:
['Date', 'Country ', 'Technology', 'SiteName', 'Volume']

Data types:
Date          object
Country       object
Technology    object
SiteName      object
Volume        object
dtype: object


################################################################################
BASIC STRUCTURE CHECK: wind_plants.csv
################################################################################
BASIC STRUCTURE CHECK

Shape:
(957, 5)

First 5 rows:


,Date,Country,Technology,SiteName,Volume
0,01/01/2024,GB,Wind,Hornsea-1,260.166079
1,02/01/2024,GB,Wind,Hornsea-1,709.480820
2,03/01/2024,GB,Wind,Hornsea-1,431.527680
3,04/01/2024,GB,Wind,Hornsea-1,223.868472
4,05/01/2024,GB,Wind,Hornsea-1,686.985009



Last 5 rows:


,Date,Country,Technology,SiteName,Volume
952,21/04/2025,GB,Wind,Hornsea-2,711.231619
953,22/04/2025,GB,Wind,Hornsea-2,808.534585
954,23/04/2025,GB,Wind,Hornsea-2,142.450340
955,24/04/2025,GB,Wind,Hornsea-2,392.082184
956,25/04/2025,GB,Wind,Hornsea-2,920.847877



Column names:
['Date', 'Country ', 'Technology', 'SiteName', 'Volume']

Data types:
Date           object
Country        object
Technology     object
SiteName       object
Volume        float64
dtype: object


# Missing and duplicate check
I performed this check to identify missing values, fully empty rows, and duplicate rows across the datasets, as these issues can affect both the quality of `database.csv` and the accuracy of the aggregation outputs.

This check showed that `gas_fr_plants.csv` contains 481 fully empty rows and 480 duplicate rows, which appear to be caused mainly by repeated empty rows in the dataset.The dataset also contains 6 missing `Volume` values except fully empty rows in valid plant records. `gas_plants.csv` contains 3 missing `Volume` values and no duplicate rows, while `wind_plants.csv` does not contain missing values or duplicate rows in this check.

These issues need to be handled before loading data into the database. Fully empty rows do not represent valid plant records, so the pipeline  will remove them. Duplicate rows can inflate aggregation results, so exact duplicates should also be dropped. Missing `Volume` values are handled by filling them with `0'.


In [9]:
def missing_and_duplicate_check(df):
    
    print("MISSING VALUES AND DUPLICATE CHECK")
    print("=" * 50)
    
    print("\nMissing values per column:")
    print(df.isna().sum())
    
    empty_rows = df.isna().all(axis=1).sum()
    print("\nFully empty rows:")
    print(empty_rows)
    
    duplicate_rows = df.duplicated().sum()
    print("\nExact duplicate rows:")
    print(duplicate_rows)

In [10]:
for name, df in datasets.items():
    print("\n")
    print("#" * 80)
    print(f"Missing and Dulicate CHECK: {name}")
    print("#" * 80)
    
    missing_and_duplicate_check(df)



################################################################################
Missing and Dulicate CHECK: gas_fr_plants.csv
################################################################################
MISSING VALUES AND DUPLICATE CHECK

Missing values per column:
Date          481
Country       481
Technology    481
SiteName      481
Volume        487
dtype: int64

Fully empty rows:
481

Exact duplicate rows:
480


################################################################################
Missing and Dulicate CHECK: gas_plants.csv
################################################################################
MISSING VALUES AND DUPLICATE CHECK

Missing values per column:
Date          0
Country       0
Technology    0
SiteName      0
Volume        3
dtype: int64

Fully empty rows:
0

Exact duplicate rows:
0


################################################################################
Missing and Dulicate CHECK: wind_plants.csv
######################################

In [11]:
def standardised_copy(df):
    
    df_check = df.copy()
    df_check.columns = df_check.columns.str.strip()
    
    return df_check

# Categorical value check
I performed this check to understand the categorical values in each dataset and to identify any formatting issues such as extra whitespace.

The `gas_fr_plants.csv` dataset contains gas technology data for one site, `Blenod-5`, in France. The `gas_plants.csv` dataset contains gas technology data for two Great Britain sites, `Pembroke-1` and `Pembroke-2`. The `wind_plants.csv` dataset contains wind technology data for two Great Britain sites, `Hornsea-1` and `Hornsea-2`. This check also showed that the `Technology` value in the wind dataset contains trailing whitespace.

The pipeline handles this by stripping whitespace from text columns and mapping country codes to full country names, such as `FR` to `France` and `GB` to `Great Britain`.

In [13]:
def categorical_value_check(df):
    
    print("CATEGORICAL VALUE CHECK")
    print("=" * 50)
    
    df_check = standardised_copy(df)
    
    categorical_cols = ["Country", "SiteName", "Technology"]
    
    for col in categorical_cols:
        if col in df_check.columns:
            print(f"\nUnique values in {col}:")
            print(df_check[col].dropna().unique())
            
            if df_check[col].dtype == "object":
                print(f"\nUnique values in {col} after stripping whitespace:")
                print(df_check[col].dropna().astype(str).str.strip().unique())
        else:
            print(f"\nColumn missing: {col}")

In [14]:
for name, df in datasets.items():
    print("\n")
    print("#" * 80)
    print(f"CATEGORICAL VALUE CHECK: {name}")
    print("#" * 80)
    
    categorical_value_check(df)



################################################################################
CATEGORICAL VALUE CHECK: gas_fr_plants.csv
################################################################################
CATEGORICAL VALUE CHECK

Unique values in Country:
['FR']

Unique values in Country after stripping whitespace:
['FR']

Unique values in SiteName:
['Blenod-5']

Unique values in SiteName after stripping whitespace:
['Blenod-5']

Unique values in Technology:
['Gas']

Unique values in Technology after stripping whitespace:
['Gas']


################################################################################
CATEGORICAL VALUE CHECK: gas_plants.csv
################################################################################
CATEGORICAL VALUE CHECK

Unique values in Country:
['GB']

Unique values in Country after stripping whitespace:
['GB']

Unique values in SiteName:
['Pembroke-1' 'Pembroke-2']

Unique values in SiteName after stripping whitespace:
['Pembroke-1' 'Pembroke-2']


# Date value check
I ran this check to validate the `Date` column and understand the date range covered by each dataset. 
The check showed that `gas_fr_plants.csv` has 481 invalid or missing date values, which are caused by the fully empty rows. Apart from these empty rows, the datasets do not contain invalid or null date values. The date range across the files is from `2024-01-01` to `2025-04-25`.
The `Date` column is important because it is required to uniquely identify daily plant records and to calculate quarterly aggregations. In the pipeline, the `Date` column is converted to datetime format, and rows where the date cannot be parsed are logged in `Logs.csv` .

In [16]:
def date_value_check(df, date_col="Date"):
    
    print("DATE VALUE CHECK")
    print("=" * 50)

    df_check = standardised_copy(df)

    if date_col not in df_check.columns:
        print(f"Date column missing: {date_col}")
        return

    parsed_dates = pd.to_datetime(
        df_check[date_col],
        errors="coerce",
        dayfirst=True
    )

    invalid_date_mask = parsed_dates.isna()

    print("\nRows with missing or invalid Date values:")
    print(invalid_date_mask.sum())

    print("\nValid date range:")
    print(parsed_dates.min(), "to", parsed_dates.max())

    if invalid_date_mask.sum() > 0:
        print("\nRows where Date is missing or invalid:")
        display(df_check.loc[invalid_date_mask])
    else:
        print("\nNo missing or invalid Date values found in existing rows.")

In [17]:
for name, df in datasets.items():
    print("\n")
    print("#" * 80)
    print(f"DATE CHECK: {name}")
    print("#" * 80)
    
    date_value_check(df)



################################################################################
DATE CHECK: gas_fr_plants.csv
################################################################################
DATE VALUE CHECK

Rows with missing or invalid Date values:
481

Valid date range:
2024-01-01 00:00:00 to 2025-04-25 00:00:00

Rows where Date is missing or invalid:


,Date,Country,Technology,SiteName,Volume
481,NaN,NaN,NaN,NaN,NaN
482,NaN,NaN,NaN,NaN,NaN
483,NaN,NaN,NaN,NaN,NaN
484,NaN,NaN,NaN,NaN,NaN
485,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...
957,NaN,NaN,NaN,NaN,NaN
958,NaN,NaN,NaN,NaN,NaN
959,NaN,NaN,NaN,NaN,NaN
960,NaN,NaN,NaN,NaN,NaN




################################################################################
DATE CHECK: gas_plants.csv
################################################################################
DATE VALUE CHECK

Rows with missing or invalid Date values:
0

Valid date range:
2024-01-01 00:00:00 to 2025-04-25 00:00:00

No missing or invalid Date values found in existing rows.


################################################################################
DATE CHECK: wind_plants.csv
################################################################################
DATE VALUE CHECK

Rows with missing or invalid Date values:
0

Valid date range:
2024-01-01 00:00:00 to 2025-04-25 00:00:00

No missing or invalid Date values found in existing rows.


# Date Sequence check
I ran this check to validate whether each plant site has a continuous daily date sequence within its available date range.

All sites have complete daily sequence except Hornsea-2 in wind_plants.csv, where 5 dates are missing from 2024-07-10 to 2024-07-14.

Missing daily records can affect quarterly statistics and country-level totals. In the pipeline, these missing dates are logged in `Logs.csv` for review. I do not create or populate new rows for the missing dates because the actual production volume for those dates is unknown.

In [19]:
def date_sequence_completeness_check(df):
    
    print("DATE SEQUENCE COMPLETENESS CHECK")
    print("=" * 50)

    df_check = standardised_copy(df)

    required_cols = ["Date", "SiteName"]
    missing_cols = [col for col in required_cols if col not in df_check.columns]

    if missing_cols:
        print(f"Missing columns for date sequence check: {missing_cols}")
        return

    df_check["Date_parsed"] = pd.to_datetime(
        df_check["Date"],
        errors="coerce",
        dayfirst=True
    )

    # Remove rows where Date itself is invalid/missing
    df_check = df_check.dropna(subset=["Date_parsed"])

    for site, group in df_check.groupby("SiteName"):
        full_date_range = pd.date_range(
            start=group["Date_parsed"].min(),
            end=group["Date_parsed"].max(),
            freq="D"
        )

        existing_dates = pd.DatetimeIndex(group["Date_parsed"].unique())
        missing_dates = full_date_range.difference(existing_dates)

        print(f"\n{site}")
        print(f"Date range: {group['Date_parsed'].min().date()} to {group['Date_parsed'].max().date()}")
        print(f"Expected daily records: {len(full_date_range)}")
        print(f"Actual daily records: {group['Date_parsed'].nunique()}")
        print(f"Missing dates: {len(missing_dates)}")

        if len(missing_dates) > 0:
            print(missing_dates.strftime("%Y-%m-%d").tolist())

In [20]:
for name, df in datasets.items():
    print("\n")
    print("#" * 80)
    print(f"DATE Sequence CHECK: {name}")
    print("#" * 80)
    
    date_sequence_completeness_check(df)



################################################################################
DATE Sequence CHECK: gas_fr_plants.csv
################################################################################
DATE SEQUENCE COMPLETENESS CHECK

Blenod-5
Date range: 2024-01-01 to 2025-04-25
Expected daily records: 481
Actual daily records: 481
Missing dates: 0


################################################################################
DATE Sequence CHECK: gas_plants.csv
################################################################################
DATE SEQUENCE COMPLETENESS CHECK

Pembroke-1
Date range: 2024-01-01 to 2025-04-25
Expected daily records: 481
Actual daily records: 481
Missing dates: 0

Pembroke-2
Date range: 2024-01-01 to 2025-04-25
Expected daily records: 481
Actual daily records: 481
Missing dates: 0


################################################################################
DATE Sequence CHECK: wind_plants.csv
#####################################################

# Volume Check
I ran this check to investigate the Volume column in more detail, as earlier checks showed missing values and data type issues. Since Volume is the main numerical field used in the final aggregations, I checked for missing values, invalid string values, negative values, and potential outliers.

I found that in gas_fr_plants dataset there are 487 null values in the volume column out of which 481 belongs to fully empty rows and remaining 6 are rows where volume is missing, there are no negative or string values in the volume column as well as no outliers. 

For the gas_plants dataset there are 3 missing volume values as well as 1 invalid string volume value = 'naa' which is probably the reason why the datatype check was showing volume column as object type earlier, there are no negative values or outliers present.

In the wind_plants dataset there are no null or string values in volume column but there are 4 negative volume values as well as 3 high volume outliers continous from 13-07-2025 to 15-07-2024 which could affect the values of the aggregations. 

The pipeline solves the above issues by dropping the fully empty values ,replacing negative, string and null volume values with 0 and logging the outliers in the logs csv file and not adding them to the final database. The outliers are really large in values compared to the average values of volumes in the wind_plants dataset and would also alter the aggregation values for quarters and countries, that is why I decided to not add them in the final database csv file.

In [22]:
def volume_check(df, volume_col="Volume"):
    
    print("VOLUME CHECK")
    print("=" * 50)
    
    df_check = standardised_copy(df)
    
    if volume_col not in df_check.columns:
        print(f"Volume column missing: {volume_col}")
        return
    
    original_volume = df_check[volume_col]
    numeric_volume = pd.to_numeric(original_volume, errors="coerce")
    
    missing_volume = original_volume.isna().sum()
    invalid_string_mask = numeric_volume.isna() & original_volume.notna()
    invalid_string_count = invalid_string_mask.sum()
    negative_volume_count = (numeric_volume < 0).sum()
    
    print("\nOriginal Volume dtype:")
    print(original_volume.dtype)
    
    print("\nMissing Volume values:")
    print(missing_volume)
    
    print("\nInvalid string Volume values:")
    print(invalid_string_count)
    
    if invalid_string_count > 0:
        print("\nInvalid Volume values found:")
        print(original_volume[invalid_string_mask].unique())
        
        print("\nRows with invalid string Volume values:")
        display(df_check.loc[invalid_string_mask])
    
    print("\nNegative Volume values:")
    print(negative_volume_count)
    
    if negative_volume_count > 0:
        print("\nRows with negative Volume:")
        display(df_check.loc[numeric_volume < 0])
    
    print("\nVolume summary after safe numeric conversion:")
    display(numeric_volume.describe())
    
    # Outlier check only on non-null and non-negative values
    valid_numeric_volume = numeric_volume.dropna()
    valid_numeric_volume = valid_numeric_volume[valid_numeric_volume >= 0]
    
    if len(valid_numeric_volume) > 0:
        q1 = valid_numeric_volume.quantile(0.25)
        q3 = valid_numeric_volume.quantile(0.75)
        iqr = q3 - q1
        upper_bound = q3 + 3 * iqr
        
        outlier_mask = numeric_volume > upper_bound
        
        print("\nPotential high-volume outliers using Q3 + 3*IQR:")
        print("Upper bound:", upper_bound)
        print("Outlier count:", outlier_mask.sum())
        
        if outlier_mask.sum() > 0:
            display(df_check.loc[outlier_mask])

In [23]:
for name, df in datasets.items():
    print("\n")
    print("#" * 80)
    print(f"Volume CHECK: {name}")
    print("#" * 80)
    
    volume_check(df)



################################################################################
Volume CHECK: gas_fr_plants.csv
################################################################################
VOLUME CHECK

Original Volume dtype:
float64

Missing Volume values:
487

Invalid string Volume values:
0

Negative Volume values:
0

Volume summary after safe numeric conversion:


count     475.000000
mean     5009.648421
std      1151.503508
min      3059.000000
25%      4078.500000
50%      5051.000000
75%      5929.500000
max      6985.000000
Name: Volume, dtype: float64


Potential high-volume outliers using Q3 + 3*IQR:
Upper bound: 11482.5
Outlier count: 0


################################################################################
Volume CHECK: gas_plants.csv
################################################################################
VOLUME CHECK

Original Volume dtype:
object

Missing Volume values:
3

Invalid string Volume values:
1

Invalid Volume values found:
['naa']

Rows with invalid string Volume values:


,Date,Country,Technology,SiteName,Volume
928,23/03/2025,GB,Gas,Pembroke-2,naa



Negative Volume values:
0

Volume summary after safe numeric conversion:


...

,Date,Country,Technology,SiteName,Volume
429,05/03/2025,GB,Wind,Hornsea-1,-27.560000
454,30/03/2025,GB,Wind,Hornsea-1,-325.210000
484,04/01/2024,GB,Wind,Hornsea-2,-550.307927
485,05/01/2024,GB,Wind,Hornsea-2,-523.899847



Volume summary after safe numeric conversion:


count       957.000000
mean        654.104050
std        4058.265477
min        -550.307927
25%         247.140967
50%         486.007385
75%         736.814133
max      123731.600000
Name: Volume, dtype: float64


Potential high-volume outliers using Q3 + 3*IQR:
Upper bound: 2192.0195003999997
Outlier count: 3


,Date,Country,Technology,SiteName,Volume
194,13/07/2024,GB,Wind,Hornsea-1,15324.72
195,14/07/2024,GB,Wind,Hornsea-1,17342.61
196,15/07/2024,GB,Wind,Hornsea-1,123731.60


# Power Plant Class Documentation
The PowerPlants class implements the full data pipeline for the three plant files. The pipeline reads raw CSV files, cleans data quality issues, logs the issues found, saves cleaned records into database.csv, retrieves the latest data, and produces the required aggregations.

The data flow is:

Raw CSV file → analyse_plant_data() → load_new_data_from_file() → save_new_data() → database.csv → get_data_from_database() → final aggregations 

## Logging Function 

We created a _write_log function which will create a Logs.csv file to store and keep an audit trail of the errors found in the datasets. Each log entry records the log time, dataset name, check name, number of affected rows, issue description and action taken. 

This helps in making the cleaning process transparent and allows rejected or changed records to be reviewed later. 

## analyse_plant_data 

This function reads the raw csv files and applies the data cleaning and validation checks which we discovered during EDA. 

It handles the following checks:

1) whitespace in column names
2) whitespace in text values
3) fully empty rows
4) invalid or missing dates
5) missing or invalid Volume values
6) negative Volume values
7) duplicate rows
8) country code mapping
9) missing date sequence logging
10) high-volume outlier logging and removal

Missing or invalid values are replaced with 0. Negative values are also replaced with 0 because production values cannot be negative. Potential high volume outliers are logged and removed from the data.

## load_new_data_from_file

This function prepares the cleaned data from analyse_plant_data function for saving into database.csv. 

It renames columns into expected database format and add two columns required by the format:

1) Updatedby: used to store the identity of the pipeline/user that loaded the data
2) Updatetime: records the time when the data is loaded

These fields in the database allows to track when each version of records was added.

## save_new_data

This function saves the prepared data into database.csv. 

The data is treated as append only, meaning each time the data is saved as new rows instead of overwriting previous rows, preserving the history of previous saves. 

Before saving the function checks that all required columns are present.

## get_data_from_database

This function reads database.csv adn returns the latest version of each plant date-record. 

Because the database is append-only, the same plant-date record may exist more than once with different update times. The latest record is selected with latest updatetime using this combination:

date + country + SiteName + Technology

## aggregate_data_to_quarterly

This function uses the latest database view and calculates quarterly summary statistics for each plant. 

It calculates and returns 
1) Mean Volume
2) Median Volume
3) Standard Deviation of Volume

The function separates quarters by years.

## aggregate_data_to_country

This function uses the latest database data and calculates the total volume grouped by country and Technology. 

This produces the required country level production summary 

In [25]:
class PowerPlants(object):
    def __init__(self):
        self.database_file = "database.csv"
        self.log_file = "logs.csv"
        self.logged_by = "petroineos"

    def _write_log(self, file_name, check_name, issue_count, issue_description, action_taken):
        

        log_entry = pd.DataFrame([{
            "log_time": datetime.now(),
            "logged_by": self.logged_by,
            "file_name": file_name,
            "check_name": check_name,
            "issue_count": issue_count,
            "issue_description": issue_description,
            "action_taken": action_taken
        }])

        if os.path.exists(self.log_file):
            existing_logs = pd.read_csv(self.log_file)
            logs = pd.concat([existing_logs, log_entry], ignore_index=True)
            logs.to_csv(self.log_file, index=False)
        else:
            log_entry.to_csv(self.log_file, index=False)

    def analyse_plant_data(self, file_path: str):
        

        df = pd.read_csv(file_path)

        file_name = os.path.basename(file_path)

        
        # Strip column names
        
        original_columns = df.columns.tolist()
        stripped_columns = [col.strip() for col in df.columns]

        if original_columns != stripped_columns:
            self._write_log(
                file_name=file_name,
                check_name="column_whitespace",
                issue_count=sum([old != new for old, new in zip(original_columns, stripped_columns)]),
                issue_description="One or more column names contained leading/trailing whitespace.",
                action_taken="Stripped whitespace from column names."
            )

        df.columns = stripped_columns

        
        # Remove fully empty rows
        
        fully_empty_rows = df.isna().all(axis=1).sum()

        if fully_empty_rows > 0:
            self._write_log(
                file_name=file_name,
                check_name="fully_empty_rows",
                issue_count=fully_empty_rows,
                issue_description="Rows found where all columns were missing.",
                action_taken="Removed fully empty rows."
            )

            df = df.dropna(how="all")
        else:
            pass

        
        # Strip whitespace from text values
        
        text_columns = ["Country", "Technology", "SiteName"]

        for col in text_columns:
            if col in df.columns:
                original_values = df[col].copy()
                stripped_values = df[col].astype(str).str.strip()

                changed_count = (original_values.astype(str) != stripped_values).sum()

                if changed_count > 0:
                    self._write_log(
                        file_name=file_name,
                        check_name=f"{col}_value_whitespace",
                        issue_count=changed_count,
                        issue_description=f"Values in {col} contained leading/trailing whitespace.",
                        action_taken=f"Stripped whitespace from {col} values."
                    )

                df[col] = stripped_values
            else:
                pass

        
        #  Convert Date to datetime and remove invalid dates
        
        if "Date" in df.columns:
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

            invalid_dates = df["Date"].isna().sum()

            if invalid_dates > 0:
                self._write_log(
                    file_name=file_name,
                    check_name="invalid_or_missing_dates",
                    issue_count=invalid_dates,
                    issue_description="Rows found with missing or invalid Date values.",
                    action_taken="Removed rows with missing or invalid Date."
                )

                df = df.dropna(subset=["Date"])
            else:
                pass
        else:
            self._write_log(
                file_name=file_name,
                check_name="missing_date_column",
                issue_count=1,
                issue_description="Date column is missing from the dataset.",
                action_taken="No cleaning applied. This is a critical issue."
            )
            raise ValueError("Date column is missing.")

        
        #  Convert Volume to numeric and handle invalid values
        
        if "Volume" in df.columns:
            original_volume = df["Volume"].copy()
            numeric_volume = pd.to_numeric(original_volume, errors="coerce")

            invalid_volume_mask = numeric_volume.isna() & original_volume.notna()
            invalid_volume_count = invalid_volume_mask.sum()

            if invalid_volume_count > 0:
                invalid_values = original_volume[invalid_volume_mask].unique()

                self._write_log(
                    file_name=file_name,
                    check_name="invalid_volume_values",
                    issue_count=invalid_volume_count,
                    issue_description=f"Non-numeric Volume values found: {invalid_values}",
                    action_taken="Converted invalid Volume values to NaN, then filled with 0."
                )
            else:
                pass

            missing_volume_count = original_volume.isna().sum()

            if missing_volume_count > 0:
                self._write_log(
                    file_name=file_name,
                    check_name="missing_volume_values",
                    issue_count=missing_volume_count,
                    issue_description="Missing Volume values found.",
                    action_taken="Filled missing Volume values with 0."
                )
            else:
                pass

            df["Volume"] = numeric_volume
            df["Volume"] = df["Volume"].fillna(0)

            negative_volume_count = (df["Volume"] < 0).sum()

            if negative_volume_count > 0:
                self._write_log(
                    file_name=file_name,
                    check_name="negative_volume_values",
                    issue_count=negative_volume_count,
                    issue_description="Negative Volume values found.",
                    action_taken="Replaced negative Volume values with 0."
                )

                def fix_negative_volume(value):
                    if value < 0:
                        return 0
                    else:
                        return value
                
                df["Volume"] = df["Volume"].apply(fix_negative_volume)
            else:
                pass

        else:
            self._write_log(
                file_name=file_name,
                check_name="missing_volume_column",
                issue_count=1,
                issue_description="Volume column is missing from the dataset.",
                action_taken="No cleaning applied. This is a critical issue."
            )
            raise ValueError("Volume column is missing.")

        
        #  Remove duplicate rows
        
        duplicate_rows = df.duplicated().sum()

        if duplicate_rows > 0:
            self._write_log(
                file_name=file_name,
                check_name="duplicate_rows",
                issue_count=duplicate_rows,
                issue_description="Exact duplicate rows found.",
                action_taken="Removed duplicate rows."
            )

            df = df.drop_duplicates()
        else:
            pass

        
        #  Country mapping
        
        if "Country" in df.columns:
            country_map = {
                "FR": "France",
                "GB": "Great Britain",
                "UK": "Great Britain"
            }

            countries_to_map = df["Country"].isin(country_map.keys()).sum()

            if countries_to_map > 0:
                self._write_log(
                    file_name=file_name,
                    check_name="country_code_mapping",
                    issue_count=countries_to_map,
                    issue_description="Country codes found instead of full country names.",
                    action_taken="Mapped country codes to full country names."
                )

                df["Country"] = df["Country"].replace(country_map)
            else:
                pass
        else:
            self._write_log(
                file_name=file_name,
                check_name="missing_country_column",
                issue_count=1,
                issue_description="Country column is missing from the dataset.",
                action_taken="No cleaning applied. This is a critical issue."
            )
            raise ValueError("Country column is missing.")

        
        # Missing date sequence check
        
        for site, group in df.groupby("SiteName"):
            full_date_range = pd.date_range(
                start=group["Date"].min(),
                end=group["Date"].max(),
                freq="D"
            )

            existing_dates = pd.DatetimeIndex(group["Date"].unique())
            missing_dates = full_date_range.difference(existing_dates)

            if len(missing_dates) > 0:
                self._write_log(
                    file_name=file_name,
                    check_name="missing_date_sequence",
                    issue_count=len(missing_dates),
                    issue_description=f"Missing daily records for {site}: {missing_dates.strftime('%Y-%m-%d').tolist()}",
                    action_taken="Logged only. Missing date rows were not created because actual Volume is unknown."
                )
            else:
                pass

        
        # Outlier check
        
        q1 = df["Volume"].quantile(0.25)
        q3 = df["Volume"].quantile(0.75)
        iqr = q3 - q1
        upper_bound = q3 + 3 * iqr
        
        outlier_mask = df["Volume"] > upper_bound
        outlier_count = outlier_mask.sum()
        
        if outlier_count > 0:
            outlier_rows = df.loc[outlier_mask].copy()
        
            self._write_log(
                file_name=file_name,
                check_name="possible_volume_outliers",
                issue_count=outlier_count,
                issue_description=(
                    f"Volume values found above outlier threshold: {upper_bound}. "
                    f"Outlier rows: {outlier_rows.to_dict(orient='records')}"
                ),
                action_taken="Removed outlier rows from cleaned data before saving to database."
            )
        
            df = df.loc[~outlier_mask].copy()
        else:
            pass

        
        # standard column order
        
        df = df[["Date", "Country", "Technology", "SiteName", "Volume"]]

        print(f"{file_name} analysed and cleaned successfully. Final shape: {df.shape}")

        return df

    
    def load_new_data_from_file(self, input_data: pd.DataFrame):
        
    
        data = input_data.copy()
    
        data = data.rename(columns={
            "Date": "date",
            "Country": "country"
        })
    
        data["updatedby"] = self.logged_by
        data["updatetime"] = datetime.now()
    
        database_columns = [
            "date",
            "country",
            "SiteName",
            "Technology",
            "updatedby",
            "updatetime",
            "Volume"
        ]
    
        data = data[database_columns]
    
        print(f"Data prepared for database load. Shape: {data.shape}")

        return data

    
    def save_new_data(self, input_data: pd.DataFrame):
        
        data = input_data.copy()

        expected_columns = [
            "date",
            "country",
            "SiteName",
            "Technology",
            "updatedby",
            "updatetime",
            "Volume"
        ]

        missing_columns = []

        for col in expected_columns:
            if col not in data.columns:
                missing_columns.append(col)
        
        if len(missing_columns) > 0:
            raise ValueError(f"Cannot save data. Missing columns: {missing_columns}")
    
        
        data = data[expected_columns]
    
        
        if os.path.exists(self.database_file):
            existing_data = pd.read_csv(self.database_file)
    
            combined_data = pd.concat(
                [existing_data, data],
                ignore_index=True
            )
    
            combined_data.to_csv(self.database_file, index=False)
    
            print(
                f"Appended {len(data)} rows to {self.database_file}. "
                f"Database now contains {len(combined_data)} rows."
            )

        else:
            data.to_csv(self.database_file, index=False)
    
            print(
                f"Created {self.database_file} and saved {len(data)} rows."
            )


    
    def get_data_from_database(self):
        
    
        if not os.path.exists(self.database_file):
            raise FileNotFoundError(
                f"{self.database_file} does not exist. Please save data before reading from the database."
            )
    
        data = pd.read_csv(self.database_file)
    
        expected_columns = [
            "date",
            "country",
            "SiteName",
            "Technology",
            "updatedby",
            "updatetime",
            "Volume"
        ]

        missing_columns = []

        for col in expected_columns:
            if col not in data.columns:
                missing_columns.append(col)
        
        if len(missing_columns) > 0:
            raise ValueError(f"Database file is missing expected columns: {missing_columns}")
    
        
        data["country"] = data["country"].astype(str).str.strip()
        data["SiteName"] = data["SiteName"].astype(str).str.strip()
        data["Technology"] = data["Technology"].astype(str).str.strip()
    
        data["date"] = pd.to_datetime(data["date"], errors="coerce", format="mixed")
        data["updatetime"] = pd.to_datetime(data["updatetime"], errors="coerce", format = "mixed")
        data["Volume"] = pd.to_numeric(data["Volume"], errors="coerce")
    
        
        data = data.dropna(
            subset=["date", "country", "SiteName", "Technology", "updatetime"]
        )
    
        key_columns = [
            "date",
            "country",
            "SiteName",
            "Technology"
        ]

        
    
        
        latest_indices = data.groupby(key_columns)["updatetime"].idxmax()
    
        latest_data = data.loc[latest_indices].copy()
    
        latest_data = latest_data.sort_values(
            by=["country", "Technology", "SiteName", "date"]
        ).reset_index(drop=True)
    
        
    
        
    
        return latest_data    
        
    def aggregate_data_to_quarterly(self):

        data = self.get_data_from_database().copy()
    
        data["date"] = pd.to_datetime(data["date"], errors="coerce", format="mixed")
        data["Volume"] = pd.to_numeric(data["Volume"], errors="coerce")
    
        data = data.dropna(subset=["date", "Volume"])
    
        data["year"] = data["date"].dt.year
        data["quarter"] = data["date"].dt.quarter
    
        quarter_map = {
            1: "Jan-Mar",
            2: "Apr-Jun",
            3: "Jul-Sep",
            4: "Oct-Dec"
        }
    
        data["quarter_label"] = data["quarter"].map(quarter_map)
        data["year_quarter"] = data["year"].astype(str) + " " + data["quarter_label"]
    
        quarterly_data = (
            data
            .groupby(["year", "quarter", "year_quarter", "SiteName"])
            .agg(
                mean_volume=("Volume", "mean"),
                median_volume=("Volume", "median"),
                std_volume=("Volume", "std")
            )
            .reset_index()
        )
    
        quarterly_wide = quarterly_data.pivot(
            index=["year", "quarter", "year_quarter"],
            columns="SiteName",
            values=["mean_volume", "median_volume", "std_volume"]
        )
    
        quarterly_wide.columns = [
            f"{site}_{metric.replace('_volume', '')}"
            for metric, site in quarterly_wide.columns
        ]
    
        site_order = sorted(data["SiteName"].unique())
        metric_order = ["mean", "median", "std"]
    
        ordered_columns = []
    
        for site in site_order:
            for metric in metric_order:
                col_name = f"{site}_{metric}"
                if col_name in quarterly_wide.columns:
                    ordered_columns.append(col_name)
    
        quarterly_wide = quarterly_wide[ordered_columns]
    
        quarterly_wide = quarterly_wide.reset_index()
        quarterly_wide = quarterly_wide.sort_values(["year", "quarter"])
    
        quarterly_wide = quarterly_wide.drop(columns=["year", "quarter"])
        quarterly_wide = quarterly_wide.reset_index(drop=True)
    
        return quarterly_wide
        

    
    def aggregate_data_to_country(self):
        
    
        data = self.get_data_from_database().copy()
    
        
        data["Volume"] = pd.to_numeric(data["Volume"], errors="coerce")
    
        
        data = data.dropna(subset=["Volume"])

        country_data = (
            data
            .groupby(["country", "Technology"], as_index=False)
            .agg(
                Volume=("Volume", "sum")
            )
        )
    
        country_data = country_data.sort_values(
            by=["country", "Technology"]
        ).reset_index(drop=True)
    
        
    
        return country_data



In [26]:
pp = PowerPlants()
new_data = pp.load_new_data_from_file(pp.analyse_plant_data("gas_fr_plants.csv"))
pp.save_new_data(new_data)
new_data = pp.load_new_data_from_file(pp.analyse_plant_data("gas_plants.csv"))
pp.save_new_data(new_data)
new_data = pp.load_new_data_from_file(pp.analyse_plant_data("wind_plants.csv"))
pp.save_new_data(new_data)

gas_fr_plants.csv analysed and cleaned successfully. Final shape: (481, 5)
Data prepared for database load. Shape: (481, 7)
Created database.csv and saved 481 rows.
gas_plants.csv analysed and cleaned successfully. Final shape: (962, 5)
Data prepared for database load. Shape: (962, 7)
Appended 962 rows to database.csv. Database now contains 1443 rows.
wind_plants.csv analysed and cleaned successfully. Final shape: (954, 5)
Data prepared for database load. Shape: (954, 7)
Appended 954 rows to database.csv. Database now contains 2397 rows.


In [27]:
latest_data = pp.get_data_from_database()
display(latest_data)

,date,country,SiteName,Technology,updatedby,updatetime,Volume
0,2024-01-01,France,Blenod-5,Gas,petroineos,2026-06-05 00:53:47.997315,6753.000000
1,2024-01-02,France,Blenod-5,Gas,petroineos,2026-06-05 00:53:47.997315,3896.000000
2,2024-01-03,France,Blenod-5,Gas,petroineos,2026-06-05 00:53:47.997315,3636.000000
3,2024-01-04,France,Blenod-5,Gas,petroineos,2026-06-05 00:53:47.997315,5138.000000
4,2024-01-05,France,Blenod-5,Gas,petroineos,2026-06-05 00:53:47.997315,5265.000000
...,...,...,...,...,...,...,...
2392,2025-04-21,Great Britain,Hornsea-2,Wind,petroineos,2026-06-05 00:53:48.024861,711.231619
2393,2025-04-22,Great Britain,Hornsea-2,Wind,petroineos,2026-06-05 00:53:48.024861,808.534585
2394,2025-04-23,Great Britain,Hornsea-2,Wind,petroineos,2026-06-05 00:53:48.024861,142.450340
2395,2025-04-24,Great Britain,Hornsea-2,Wind,petroineos,2026-06-05 00:53:48.024861,392.082184


In [28]:
quarterly_data = pp.aggregate_data_to_quarterly()
display(quarterly_data)

,year_quarter,Blenod-5_mean,Blenod-5_median,Blenod-5_std,Hornsea-1_mean,Hornsea-1_median,Hornsea-1_std,Hornsea-2_mean,Hornsea-2_median,Hornsea-2_std,Pembroke-1_mean,Pembroke-1_median,Pembroke-1_std,Pembroke-2_mean,Pembroke-2_median,Pembroke-2_std
0,2024 Jan-Mar,4948.417582,4912.0,1054.780273,489.115071,491.417052,280.556287,481.970791,442.588117,291.367877,7118.450549,7192.0,1081.092033,6893.472527,6824.0,1188.405826
1,2024 Apr-Jun,4976.780220,4987.0,1130.818738,458.989280,466.683337,266.749336,477.136650,425.819859,282.352554,7055.450549,6828.0,1193.288652,7086.824176,7097.0,1156.965404
2,2024 Jul-Sep,5031.195652,4941.5,1141.124279,516.853089,499.571759,285.640158,482.641581,454.421450,265.213415,7210.804348,7311.5,1134.721075,7041.130435,7068.0,1146.210644
3,2024 Oct-Dec,4738.282609,5048.5,1711.083400,478.276781,500.668525,293.671778,506.687955,488.632975,285.389968,7071.673913,7294.5,1154.860232,6863.586957,6766.5,1236.674486
4,2025 Jan-Mar,5053.255556,5238.5,1215.552693,510.918435,516.975897,277.968379,530.510781,571.584016,290.506335,6816.377778,6769.0,1100.306184,6920.877778,7361.0,1828.154044
5,2025 Apr-Jun,4912.200000,4461.0,1287.277813,452.714035,406.341511,278.009828,551.854866,533.487056,316.884377,7011.880000,7006.0,1008.187124,6965.360000,7071.0,1300.411886


In [29]:
country_data = pp.aggregate_data_to_country()
display(country_data)

,country,Technology,Volume
0,France,Gas,2.379583e+06
1,Great Britain,Gas,6.741038e+06
2,Great Britain,Wind,4.710056e+05
